In [14]:
import pandas as pd
import numpy as np

In [6]:
qualifying = pd.read_csv('qualifying.csv')

In [7]:
qualifying.head()

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,01:26.6,01:25.2,01:26.7
1,2,18,9,2,4,2,01:26.1,01:25.3,01:26.9
2,3,18,5,1,23,3,01:25.7,01:25.5,01:27.1
3,4,18,13,6,2,4,01:26.0,01:25.7,01:27.2
4,5,18,2,2,3,5,01:26.0,01:25.5,01:27.2


In [8]:
qualifying_filtered = qualifying.loc[qualifying['position'] == 1,['raceId', 'constructorId', 'position']]

In [10]:
qualifying_filtered.head()

,raceId,constructorId,position
0,18,1,1
22,19,6,1
44,20,2,1
66,21,6,1
88,22,6,1


In [11]:
results = pd.read_csv('results.csv')

In [12]:
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [13]:
results_filtered = results[['raceId', 'constructorId', 'positionOrder']]

In [15]:
results_filtered['result_ind'] = np.where(results_filtered['positionOrder']==1,1,0)

/tmp/ipykernel_14205/318814858.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_filtered['result_ind'] = np.where(results_filtered['positionOrder']==1,1,0)


In [16]:
pole_and_results = qualifying_filtered.merge(results_filtered, on=['raceId','constructorId'], how='left')

In [17]:
pole_and_results.head()

,raceId,constructorId,position,positionOrder,result_ind
0,18,1,1,1,1
1,18,1,1,5,0
2,19,6,1,1,1
3,19,6,1,19,0
4,20,2,1,3,0


In [22]:
pole_and_results = pole_and_results.rename(columns={'position' : 'pole_ind'})

In [24]:
pole_and_results.head()

,raceId,constructorId,pole_ind,positionOrder,result_ind
0,18,1,1,1,1
1,18,1,1,5,0
2,19,6,1,1,1
3,19,6,1,19,0
4,20,2,1,3,0


In [25]:
pole_and_results['pole_to_win_ind'] = np.where((pole_and_results['pole_ind'] == 1) & (pole_and_results['result_ind'] == 1), 1, 0)

In [28]:
pole_and_results_agg = pole_and_results.groupby('constructorId').agg(pole_wins = ('pole_ind', 'sum'), race_wins=('result_ind','sum'),pole_to_race_wins=('pole_to_win_ind','sum')).reset_index()

In [29]:
pole_and_results_agg.head()

,constructorId,pole_wins,race_wins,pole_to_race_wins
0,1,128,31,31
1,2,2,0,0
2,3,76,18,18
3,4,40,12,12
4,5,2,1,1


In [31]:
pole_and_results_agg['conversion_rate'] = pole_and_results_agg['pole_to_race_wins']/pole_and_results_agg['pole_wins'].round(2)

In [32]:
pole_and_results_agg.sort_values(['conversion_rate'],ascending=[False])

,constructorId,pole_wins,race_wins,pole_to_race_wins,conversion_rate
4,5,2,1,1,0.500000
13,23,10,4,4,0.400000
14,131,270,101,101,0.374074
7,9,210,77,77,0.366667
3,4,40,12,12,0.300000
5,6,208,57,57,0.274038
12,22,22,6,6,0.272727
0,1,128,31,31,0.242188
2,3,76,18,18,0.236842
8,10,2,0,0,0.000000


In [33]:
constructors = pd.read_csv('constructors.csv')

In [35]:
constructors.head()

,constructorId,constructorRef,name,nationality,url
0,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren
1,2,bmw_sauber,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber
2,3,williams,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Pr...
3,4,renault,Renault,French,http://en.wikipedia.org/wiki/Renault_in_Formul...
4,5,toro_rosso,Toro Rosso,Italian,http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso


In [41]:
pole_to_win_constructors = pole_and_results_agg.merge(constructors[['name','constructorId']], on='constructorId', how='left').sort_values(['conversion_rate'],ascending=[False])

In [42]:
pole_to_win_constructors.head()

,constructorId,pole_wins,race_wins,pole_to_race_wins,conversion_rate,name
4,5,2,1,1,0.500000,Toro Rosso
13,23,10,4,4,0.400000,Brawn
14,131,270,101,101,0.374074,Mercedes
7,9,210,77,77,0.366667,Red Bull
3,4,40,12,12,0.300000,Renault
